[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/30_sequence_models/03_lstm_gru.ipynb)

# Bài 03 — LSTM và GRU

**🎯 Mục tiêu**
- **LSTM**: trạng thái ô (cell state), cổng quên / vào / ra; vì sao đường **cộng** giữ được gradient.
- **GRU**: phiên bản gọn với cổng cập nhật và cổng reset.
- Cài đặt một bước LSTM/GRU from scratch, khớp `nn.LSTMCell` / `nn.GRUCell`.
- Thí nghiệm **adding problem** (phụ thuộc xa): RNN vs LSTM vs GRU; mô hình ngôn ngữ ký tự LSTM vs RNN.
- Thực hành: hai chiều (bidirectional), xếp chồng, padding; template model.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/30_sequence_models

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ml_course.visualization import set_style

try:
    import torch
    from torch import nn
except ImportError as e:
    raise ImportError("Chương này cần PyTorch: pip install -e \".[dl]\"") from e

set_style()
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.deterministic = True
print("thiết bị:", device)

## 1. LSTM

Ngoài trạng thái ẩn $\mathbf h_t$, LSTM (Hochreiter & Schmidhuber, 1997) có **trạng thái ô** $\mathbf c_t$ — một "băng chuyền" thông tin chạy dọc thời gian, chỉ bị sửa qua các **cổng** (giá trị trong $(0, 1)$, nhân từng phần tử):

$$\begin{aligned}
\mathbf f_t &= \sigma(W_f[\mathbf x_t, \mathbf h_{t-1}] + \mathbf b_f) && \text{cổng quên: giữ bao nhiêu ký ức cũ}\\
\mathbf i_t &= \sigma(W_i[\mathbf x_t, \mathbf h_{t-1}] + \mathbf b_i) && \text{cổng vào: ghi bao nhiêu thông tin mới}\\
\mathbf g_t &= \tanh(W_g[\mathbf x_t, \mathbf h_{t-1}] + \mathbf b_g) && \text{nội dung ứng viên}\\
\mathbf o_t &= \sigma(W_o[\mathbf x_t, \mathbf h_{t-1}] + \mathbf b_o) && \text{cổng ra}\\
\mathbf c_t &= \mathbf f_t\odot\mathbf c_{t-1} + \mathbf i_t\odot\mathbf g_t\\
\mathbf h_t &= \mathbf o_t\odot\tanh(\mathbf c_t)
\end{aligned}$$

**Vì sao giữ được gradient?** $\partial\mathbf c_t/\partial\mathbf c_{t-1} = \text{diag}(\mathbf f_t)$ (cộng thêm các số hạng qua cổng) — không có nhân ma trận $W$ hay đạo hàm tanh trên đường chính. Khi $\mathbf f_t \approx 1$, gradient chảy gần như nguyên vẹn qua nhiều bước (bài 02, mục 3). Mẹo thực hành: khởi tạo **bias cổng quên = 1** để mặc định "nhớ".

In [ ]:
sigmoid = lambda z: 1 / (1 + np.exp(-z))


def lstm_step(x, h, c, W_ih, W_hh, b_ih, b_hh):
    """Theo quy ước PyTorch: các cổng xếp theo thứ tự (i, f, g, o) trong ma trận trọng số."""
    gates = W_ih @ x + b_ih + W_hh @ h + b_hh
    i, f, g, o = np.split(gates, 4)
    i, f, g, o = sigmoid(i), sigmoid(f), np.tanh(g), sigmoid(o)
    c_new = f * c + i * g
    h_new = o * np.tanh(c_new)
    return h_new, c_new


torch.manual_seed(0)
cell = nn.LSTMCell(5, 7)
W = {k: v.detach().double().numpy() for k, v in cell.state_dict().items()}
x, h, c = torch.randn(5), torch.randn(7), torch.randn(7)
h_ref, c_ref = cell(x[None], (h[None], c[None]))
h_ours, c_ours = lstm_step(x.double().numpy(), h.double().numpy(), c.double().numpy(), W["weight_ih"], W["weight_hh"], W["bias_ih"], W["bias_hh"])
print(f"LSTM: sai lệch h = {np.abs(h_ours - h_ref.detach().numpy()[0]).max():.1e}, c = {np.abs(c_ours - c_ref.detach().numpy()[0]).max():.1e}")
print(f"số tham số LSTMCell(5, 7) = {sum(p.numel() for p in cell.parameters())} = 4 × (7·5 + 7·7 + 2·7) — gấp 4 lần RNN cùng kích thước")

## 2. GRU

GRU (Cho et al., 2014) gộp trạng thái ô và trạng thái ẩn, chỉ dùng hai cổng:
$$\begin{aligned}
\mathbf r_t &= \sigma(W_r\mathbf x_t + U_r\mathbf h_{t-1} + \mathbf b_r) && \text{cổng reset}\\
\mathbf z_t &= \sigma(W_z\mathbf x_t + U_z\mathbf h_{t-1} + \mathbf b_z) && \text{cổng cập nhật}\\
\mathbf n_t &= \tanh(W_n\mathbf x_t + \mathbf b_{in} + \mathbf r_t\odot(U_n\mathbf h_{t-1} + \mathbf b_{hn})) && \text{ứng viên}\\
\mathbf h_t &= (1 - \mathbf z_t)\odot\mathbf n_t + \mathbf z_t\odot\mathbf h_{t-1}
\end{aligned}$$
$\mathbf z_t \approx 1$ → giữ nguyên trạng thái cũ (đường "tắt" theo thời gian). Ít tham số hơn LSTM (3 khối thay vì 4), hiệu quả thường tương đương.

In [ ]:
def gru_step(x, h, W_ih, W_hh, b_ih, b_hh):
    """Quy ước PyTorch: thứ tự (r, z, n)."""
    xi = W_ih @ x + b_ih
    hh = W_hh @ h + b_hh
    xr, xz, xn = np.split(xi, 3)
    hr, hz, hn = np.split(hh, 3)
    r, z = sigmoid(xr + hr), sigmoid(xz + hz)
    n = np.tanh(xn + r * hn)
    return (1 - z) * n + z * h


gcell = nn.GRUCell(5, 7)
Wg = {k: v.detach().double().numpy() for k, v in gcell.state_dict().items()}
h_ref = gcell(x[None], h[None])
h_ours = gru_step(x.double().numpy(), h.double().numpy(), Wg["weight_ih"], Wg["weight_hh"], Wg["bias_ih"], Wg["bias_hh"])
print(f"GRU: sai lệch h = {np.abs(h_ours - h_ref.detach().numpy()[0]).max():.1e} | số tham số = {sum(p.numel() for p in gcell.parameters())}")

## 3. Adding problem: phụ thuộc xa

Bài toán kinh điển (Hochreiter & Schmidhuber, 1997): chuỗi dài $T$, mỗi bước có (số ngẫu nhiên trong $[0, 1]$, cờ 0/1). Đúng **hai** bước có cờ 1 (một ở nửa đầu, một ở nửa sau); đích = **tổng hai số được đánh dấu**. Dự đoán hằng số 1 cho MSE ≈ 0.167 — mô hình phải nhớ một số từ hàng chục bước trước mới làm tốt hơn.

In [ ]:
def adding_batch(n, T, g):
    values = torch.rand(n, T, generator=g)
    marks = torch.zeros(n, T)
    a = torch.randint(0, T // 2, (n,), generator=g)
    b = torch.randint(T // 2, T, (n,), generator=g)
    marks[torch.arange(n), a] = 1
    marks[torch.arange(n), b] = 1
    return torch.stack([values, marks], dim=-1), (values * marks).sum(1)


class SeqRegressor(nn.Module):
    def __init__(self, kind, hidden=64):
        super().__init__()
        self.rnn = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[kind](2, hidden, batch_first=True)
        self.head = nn.Linear(hidden, 1)
        if kind in ("LSTM", "GRU"):                                     # cổng quên (LSTM) / cập nhật (GRU) bias = 1 → mặc định giữ trí nhớ
            with torch.no_grad():
                self.rnn.bias_ih_l0[hidden:2 * hidden].fill_(1.0)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.head(out[:, -1]).squeeze(1)


def train_adding(kind, T=50, steps=3000, seed=0):
    torch.manual_seed(seed)
    g = torch.Generator().manual_seed(seed)
    X_val, y_val = adding_batch(1000, T, torch.Generator().manual_seed(seed + 100))
    model = SeqRegressor(kind).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    curve = []
    for step in range(1, steps + 1):
        xb, yb = adding_batch(64, T, g)
        opt.zero_grad()
        nn.functional.mse_loss(model(xb.to(device)), yb.to(device)).backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        if step % 250 == 0:
            with torch.no_grad():
                curve.append(nn.functional.mse_loss(model(X_val.to(device)), y_val.to(device)).item())
    return curve


fig, ax = plt.subplots(figsize=(9, 4))
for kind in ["RNN", "LSTM", "GRU"]:
    t0 = time.perf_counter()
    curve = train_adding(kind)
    ax.semilogy(np.arange(1, len(curve) + 1) * 250, curve, marker=".", label=kind)
    print(f"{kind:5s}: MSE validation sau 3 000 bước = {curve[-1]:.4f} | {time.perf_counter() - t0:.0f}s")
ax.axhline(0.167, color="gray", ls="--", label="dự đoán hằng số (≈ 0.167)")
ax.set_xlabel("bước")
ax.set_ylabel("MSE validation")
ax.legend()
plt.show()

Với $T = 50$: RNN mắc kẹt ở mức dự đoán hằng số (không nhớ được số ở nửa đầu chuỗi); LSTM và GRU "bừng tỉnh" sau vài trăm–vài nghìn bước và gần như giải hoàn toàn. Lưu ý: kết quả loại bài toán này có tính "ngưỡng" — đổi seed, độ dài, learning rate có thể làm thời điểm bừng tỉnh thay đổi nhiều; với $T$ lớn hơn nữa, cả LSTM cũng cần nhiều bước hơn.

## 4. Mô hình ngôn ngữ ký tự: RNN vs LSTM

Lặp lại bài toán của bài 01 bằng PyTorch (embedding, batch, Adam), so sánh loss validation:

In [ ]:
reviews = pd.read_csv("../../data/sample/reviews_vi.csv")["text"].tolist()
corpus = "\n".join(reviews) + "\n"
chars = sorted(set(corpus))
stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in corpus])
split = int(0.9 * len(data))
train_data, val_data = data[:split], data[split:]


class CharLM(nn.Module):
    def __init__(self, kind, vocab, emb=32, hidden=128):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb)
        self.rnn = {"RNN": nn.RNN, "LSTM": nn.LSTM}[kind](emb, hidden, batch_first=True)
        self.head = nn.Linear(hidden, vocab)

    def forward(self, x):
        out, _ = self.rnn(self.emb(x))
        return self.head(out)


def get_batch(source, g, batch=64, seq=64):
    starts = torch.randint(0, len(source) - seq - 1, (batch,), generator=g)
    x = torch.stack([source[s:s + seq] for s in starts])
    y = torch.stack([source[s + 1:s + seq + 1] for s in starts])
    return x.to(device), y.to(device)


def train_lm(kind, steps=1500, seed=0):
    torch.manual_seed(seed)
    model = CharLM(kind, len(chars)).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=3e-3)
    g = torch.Generator().manual_seed(seed)
    for _ in range(steps):
        x, y = get_batch(train_data, g)
        opt.zero_grad()
        nn.functional.cross_entropy(model(x).reshape(-1, len(chars)), y.reshape(-1)).backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
    model.eval()
    gv = torch.Generator().manual_seed(123)
    with torch.no_grad():
        val = np.mean([nn.functional.cross_entropy(model(x).reshape(-1, len(chars)), y.reshape(-1)).item()
                       for x, y in (get_batch(val_data, gv) for _ in range(20))])
    return model, val


lms = {}
for kind in ["RNN", "LSTM"]:
    t0 = time.perf_counter()
    lms[kind], val = train_lm(kind)
    print(f"{kind:5s}: cross-entropy validation = {val:.3f} nats/ký tự (perplexity {np.exp(val):.2f}) | {time.perf_counter() - t0:.0f}s")

In [ ]:
def generate(model, start="\n", n=200, temperature=0.7, seed=0):
    g = torch.Generator().manual_seed(seed)
    x = torch.tensor([[stoi[start]]], device=device)
    state, out = None, []
    model.eval()
    with torch.no_grad():
        for _ in range(n):
            o, state = model.rnn(model.emb(x), state)
            probs = torch.softmax(model.head(o[:, -1]) / temperature, dim=-1).cpu()
            ix = torch.multinomial(probs, 1, generator=g).item()
            out.append(chars[ix])
            x = torch.tensor([[ix]], device=device)
    return "".join(out)


print(generate(lms["LSTM"]))

Trên ngữ liệu này, RNN và LSTM gần như **ngang nhau**: review được ghép từ các cụm từ ngắn, dự đoán ký tự tiếp theo chủ yếu cần ngữ cảnh vài chục ký tự — trong tầm với của RNN. Lợi thế của LSTM/GRU thể hiện khi bài toán **cần nhớ xa** (mục 3). Hãy chọn mô hình theo cấu trúc phụ thuộc của dữ liệu, và đo — đừng mặc định cái phức tạp hơn sẽ tốt hơn.

## 5. Thực hành với chuỗi

- **Bidirectional** (`bidirectional=True`): đọc chuỗi theo cả hai chiều, ghép trạng thái — tốt cho bài toán có **toàn bộ** chuỗi khi dự đoán (phân loại câu, gán nhãn từ), **không** dùng cho dự báo/sinh văn bản (không được nhìn tương lai).
- **Xếp chồng** (`num_layers=2`, `dropout=...` giữa các lớp).
- **Độ dài thay đổi**: padding + `pack_padded_sequence` để bỏ qua phần đệm; hoặc lấy trạng thái tại vị trí cuối thật của mỗi chuỗi.
- **Embedding** thay cho one-hot; có thể khởi tạo từ word2vec/fastText (chương 32).

In [ ]:
bi = nn.LSTM(16, 32, num_layers=2, batch_first=True, bidirectional=True, dropout=0.1)
out, (h_n, c_n) = bi(torch.randn(4, 10, 16))
print("đầu ra:", tuple(out.shape), "= (batch, bước, 32 × 2 chiều) | h_n:", tuple(h_n.shape), "= (2 lớp × 2 chiều, batch, 32)")

## 6. Template model: RNN / LSTM / GRU

| Mục | Nội dung |
|---|---|
| Problem | dữ liệu tuần tự: văn bản, chuỗi thời gian, âm thanh, log sự kiện |
| Intuition | đọc từng bước, cập nhật trạng thái ẩn tóm tắt quá khứ; cổng quyết định nhớ/quên |
| Math | RNN: $\mathbf h_t = \tanh(W\mathbf x_t + U\mathbf h_{t-1})$; LSTM: $\mathbf c_t = \mathbf f\odot\mathbf c_{t-1} + \mathbf i\odot\mathbf g$ |
| Algorithm | BPTT (truncated), Adam, gradient clipping |
| Complexity | $O(T\cdot H^2)$ — **tuần tự** theo thời gian, khó song song hoá |
| Hyperparameters | kích thước ẩn, số lớp, dropout, độ dài cắt BPTT, lr, clip |
| Assumptions | thông tin cần thiết nén được vào trạng thái kích thước cố định |
| Strengths | xử lý độ dài bất kỳ, bộ nhớ hằng số khi suy luận, hợp chuỗi thời gian/streaming |
| Weaknesses | khó học phụ thuộc rất xa, không song song theo thời gian, chậm với chuỗi dài |
| Failure cases | RNN thuần với phụ thuộc xa (vanishing); exploding nếu không clip |
| Production | streaming/on-device, dự báo chuỗi thời gian; NLP hiện đại chuyển sang Transformer |
| Alternatives | Transformer (chương 31), CNN 1D/TCN, state-space models (Mamba) |

## 🧠 Kiểm tra nhanh

1. Mô tả vai trò ba cổng của LSTM. Vì sao bias cổng quên nên khởi tạo dương?
2. Vì sao đường $\mathbf c_t$ giúp gradient không tắt?
3. GRU khác LSTM ở đâu? Khi nào chọn cái nào?
4. Adding problem kiểm tra năng lực gì? Vì sao RNN thất bại?
5. Khi nào **không** được dùng LSTM hai chiều?

## 📌 Tóm tắt

- LSTM: trạng thái ô + cổng quên/vào/ra; đường cộng giữ gradient; GRU gọn hơn với cổng cập nhật/reset.
- Thí nghiệm: RNN thất bại với phụ thuộc 50 bước; LSTM/GRU giải được; với dữ liệu chỉ cần ngữ cảnh ngắn, RNN và LSTM ngang nhau.
- Thực hành: embedding, bidirectional, xếp chồng, padding/packing, clipping.

## 📚 Tài liệu tham khảo

- Hochreiter & Schmidhuber (1997) — *Long Short-Term Memory*
- Cho et al. (2014) — *Learning Phrase Representations using RNN Encoder–Decoder* (GRU)
- Olah (2015) — *Understanding LSTM Networks* (colah.github.io)
- Jozefowicz, Zaremba & Sutskever (2015) — *An Empirical Exploration of Recurrent Network Architectures*